In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

## Prepare Data Set for Machine Learning

Load csv

In [2]:
from pathlib import Path
DATA_DIR = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "raw"
CSV = DATA_DIR / "Mekong_Delta_Multiple_campsites_datasets_capacity_over_200_anonymised_final.csv"

df_raw = pd.read_csv(CSV, low_memory=False, na_values=["null"])


Add additional columns, handle missing values and categorical columns

In [3]:
# add a Year column for easier time-based splitting later on

df_raw['Year'] = pd.to_datetime(df_raw['WeekStartDate']).dt.year

# extract the number of bedrooms from the "Bedrooms" column and convert it to an integer

df_raw["Bedrooms"] = df_raw["Bedrooms"].str.extract(r"(\d+)").astype(int)

# Split 'AccoTypeRangeCode' into 'AccommodationType' and 'RangeType'

df_raw[['AccommodationType', 'RangeType']] = (
    df_raw['AccoTypeRangeCode']
    .str.split(r'\s*\|\s*', n=1, expand=True)
)

# create a binary feature indicating whether the period is a special period or not

df_raw["is_special_period"] = df_raw["SpecialPeriodCode"].notna().astype(int)

Calculate the weighted average price, label the first price-data-point as 'initial price' and collapse the df so that only one row per ReservableOptionID remains

In [4]:
df_RO = df_raw.copy()
price_col = "AverageDiscPriceForWeekBeforeArrival"

# Single, unweighted target definition (matches the src pipeline / approach 2):
# InitialPrice = the discounted price at the EARLIEST priced horizon of the option.
# Non-positive prices are treated as missing so a 0 never counts as an opening price.
df_RO[price_col] = df_RO[price_col].where(df_RO[price_col] > 0)

# Collapse the 4 market groups to one row per (option, horizon): mean price.
collapsed = (
    df_RO.groupby(["ReservableOptionId", "WeekBeforeArrival"], as_index=False)
    .agg(price=(price_col, "mean"))
)

# Earliest priced horizon = highest WeekBeforeArrival that still carries a price.
priced = collapsed.dropna(subset=["price"])
opening_idx = priced.groupby("ReservableOptionId")["WeekBeforeArrival"].idxmax()
initial = (
    priced.loc[opening_idx, ["ReservableOptionId", "price"]]
    .rename(columns={"price": "InitialPrice"})
)

# One static row per option (attributes are constant within an option-week),
# attach the opening price, and drop options without a valid (> 0) one.
df_RO = (
    df_RO.drop_duplicates("ReservableOptionId")
    .merge(initial, on="ReservableOptionId", how="left")
    .dropna(subset=["InitialPrice"])
)
df_RO = df_RO[df_RO["InitialPrice"] > 0].set_index("Year")

print(f"options with a valid InitialPrice: {len(df_RO):,}")
print(df_RO["InitialPrice"].describe().round(2).to_string())


options with a valid InitialPrice: 3,353
count    3353.00
mean      114.75
std        85.37
min        28.25
25%        50.50
50%        73.66
75%       169.00
max       535.00


## Train Test Split

In [5]:
# features and target variable

features = [
    'BrandGroupCode',
    'CampsiteCode',
    'is_special_period',
    'SeasonalCluster',
    'CampsiteType',
    'AccommodationType',
    'RangeType',
    'Airco',
    'Bedrooms',
    'DeckingType',
    'Bathrooms',
    'TV',
    'Capacity',
    
]
target = ['InitialPrice']

In [6]:
# Prepare training and test data
y_train = df_RO.loc[2024]['InitialPrice'].copy()
y_test = df_RO.loc[2025]['InitialPrice'].copy()

x_train = df_RO.loc[2024, features].copy()
x_test = df_RO.loc[2025, features].copy()

In [7]:
# Pool campsites with less than 30 unique options into "Other"

unique_counts = (
    df_RO.groupby("CampsiteCode")["ReservableOptionId"]
    .nunique()
    .reset_index(name="UniqueReservableOptions")
)

keep   = unique_counts[unique_counts["UniqueReservableOptions"] >= 30]["CampsiteCode"]                      

for frame in (x_train, x_test):
    frame["CampsiteCodeGrouped"] = frame["CampsiteCode"].where(
        frame["CampsiteCode"].isin(keep), "Other"
    )

x_train.drop(columns=["CampsiteCode"], inplace=True)
x_test.drop(columns=["CampsiteCode"], inplace=True)

In [8]:
# Fill missing categoricals with the TRAIN mode (never the test set's own mode -
# that would leak test information into the features).
fill_cols = ["Airco", "TV", "DeckingType", "Bathrooms"]
train_modes = {col: x_train[col].mode(dropna=True)[0] for col in fill_cols}

for col, value in train_modes.items():
    x_train[col] = x_train[col].fillna(value)
    x_test[col]  = x_test[col].fillna(value)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_54296\705454468.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_train[col] = x_train[col].fillna(value)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_54296\705454468.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x_test[col]  = x_test[col].fillna(value)


## One-Hot-Encoding

In [9]:
# encode catergorical variables using one-hot encoding, ensuring the same columns in train and test sets

cols_to_encode = [
    "CampsiteCodeGrouped",
    "SeasonalCluster",
    "AccommodationType",
    "RangeType",
    "CampsiteType",
    "DeckingType",
    "BrandGroupCode"
]

x_train = pd.get_dummies(
    x_train,
    columns=cols_to_encode,
    drop_first=False  
)

x_test = pd.get_dummies(
    x_test,
    columns=cols_to_encode,
    drop_first=False  
)

x_test = x_test.reindex(columns=x_train.columns, fill_value=0)

## 2 Machine Learning Models

Ridge Regression

In [10]:
scaler = StandardScaler()

For Ridge Regression: Scaler

In [11]:
numeric_cols = ['Bedrooms', 'Bathrooms', 'Capacity']
x_train_scaled = x_train.copy()
x_train_scaled[numeric_cols] = scaler.fit_transform(x_train[numeric_cols])
x_test_scaled = x_test.copy()
x_test_scaled[numeric_cols] = scaler.transform(x_test[numeric_cols])

In [12]:
# 1) Fit Ridge on LOG-price, letting it choose alpha by CV on TRAIN only.
#    Prices are right-skewed, so log stabilises variance and turns the loss into a
#    multiplicative (~ percentage) error; predictions are back-transformed with expm1
#    so every metric below is on the original euro scale.
alphas = np.logspace(-3, 3, 25)            # 0.001 ... 1000
ridge = RidgeCV(alphas=alphas)
ridge.fit(x_train_scaled, np.log1p(y_train))
print(f"chosen alpha = {ridge.alpha_:.4g}")

# 2) Predict the held-out test season (back to euros)
y_pred = np.expm1(ridge.predict(x_test_scaled))

# 3) Evaluate - RMSE, MAE, R2, MAPE (MAPE guarded against zero-priced rows)
def regression_report(y_true, y_pred, label=""):
    y_true = np.asarray(y_true, dtype="float64"); y_pred = np.asarray(y_pred, dtype="float64")
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    nz   = y_true != 0
    mape = np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100
    print(f"{label:>12} | RMSE {rmse:8.2f} | MAE {mae:8.2f} | R2 {r2:6.3f} | MAPE {mape:5.1f}%")
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}

ridge_metrics = regression_report(y_test, y_pred, "Ridge")

# 4) Interpretation - which features move the opening price, and which way
coef = pd.Series(ridge.coef_, index=x_train_scaled.columns).sort_values(key=np.abs, ascending=False)
print()
print("Top 15 price drivers:")
print(coef.head(15).to_string())


chosen alpha = 1.778
       Ridge | RMSE    35.02 | MAE    25.38 | R2  0.844 | MAPE  25.1%

Top 15 price drivers:
SeasonalCluster_HS                 0.811642
SeasonalCluster_S2                -0.393463
SeasonalCluster_LS                -0.287587
SeasonalCluster_WTR               -0.266582
RangeType_Cheap                   -0.263912
CampsiteCodeGrouped_Campsite006   -0.262351
CampsiteCodeGrouped_Campsite018    0.231423
RangeType_ExtraNice                0.226348
CampsiteCodeGrouped_Other         -0.224999
CampsiteCodeGrouped_Campsite023   -0.195973
CampsiteCodeGrouped_Campsite030    0.194247
CampsiteCodeGrouped_Campsite014   -0.163783
CampsiteCodeGrouped_Campsite013   -0.157088
RangeType_Better+                  0.142336
SeasonalCluster_S1                 0.135990


Random Forest

In [13]:
# 1) Light CV search over the two knobs that control overfitting (parallel to RidgeCV's alpha).
param_grid = {
    "max_depth":        [None, 10, 20],   # how deep each tree may grow
    "min_samples_leaf": [1, 5, 10],       # min samples per leaf; higher = smoother, less overfit
}
rf_base = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
search = GridSearchCV(rf_base, param_grid, cv=5, scoring="neg_root_mean_squared_error")
search.fit(x_train, np.log1p(y_train))     # fit on log-price, like Ridge
rf = search.best_estimator_
print("best params:", search.best_params_)

# 2) Predict the held-out test season (back to euros)
y_pred_rf = np.expm1(rf.predict(x_test))

# 3) Evaluate - SAME function as Ridge, so the rows line up
rf_metrics = regression_report(y_test, y_pred_rf, "RandomForest")

# 4) Interpretation - feature importances (magnitude of reliance, NOT direction)
importances = pd.Series(rf.feature_importances_, index=x_train.columns).sort_values(ascending=False)
print()
print("Top 15 features the forest relied on:")
print(importances.head(15).to_string())


best params: {'max_depth': None, 'min_samples_leaf': 5}
RandomForest | RMSE    37.12 | MAE    25.93 | R2  0.825 | MAPE  25.0%



Top 15 features the forest relied on:
SeasonalCluster_HS                 0.721411
RangeType_Cheap                    0.071252
SeasonalCluster_S1                 0.044786
Bedrooms                           0.025743
Capacity                           0.021344
RangeType_ExtraNice                0.013901
RangeType_Chill                    0.012171
CampsiteCodeGrouped_Campsite030    0.010585
TV                                 0.010180
DeckingType_Small Decking          0.009469
DeckingType_Covered Decking        0.009031
RangeType_Chill+                   0.007306
CampsiteCodeGrouped_Campsite018    0.005356
CampsiteCodeGrouped_Campsite007    0.004298
CampsiteCodeGrouped_Campsite017    0.002978


Baseline

In [14]:
# --- Baselines: the bar the ML models must beat ---

# Baseline A — predict the global TRAIN mean price for every test option
baseline_global = np.full(len(y_test), y_train.mean())
regression_report(y_test, baseline_global, "Baseline-mean")

# Baseline B — predict the TRAIN mean price within a comparable segment
segment_cols = ["SeasonalCluster", "AccommodationType", "RangeType"]

seg_means   = df_RO.loc[2024].groupby(segment_cols)["InitialPrice"].mean()
global_mean = y_train.mean()

# map each 2025 option to its segment's 2024 mean; unseen segments -> global mean
baseline_segment = (
    df_RO.loc[2025]
    .set_index(segment_cols)
    .index.map(seg_means)
    .to_numpy(dtype="float64")
)
baseline_segment = np.where(np.isnan(baseline_segment), global_mean, baseline_segment)
regression_report(y_test, baseline_segment, "Baseline-segment")

Baseline-mean | RMSE    92.61 | MAE    81.08 | R2 -0.092 | MAPE 110.3%
Baseline-segment | RMSE    41.49 | MAE    28.48 | R2  0.781 | MAPE  26.6%


{'rmse': 41.49250099471863,
 'mae': 28.4807202934841,
 'r2': 0.7809000493967803,
 'mape': np.float64(26.64059616098165)}

Per Seasonalcluster Evaluation

In [15]:

per_cluster = pd.DataFrame({
    "SeasonalCluster": df_RO.loc[2025, "SeasonalCluster"].to_numpy(),
    "y_true":          y_test.to_numpy(),
    "Baseline-segment": baseline_segment,
    "Ridge":            y_pred,
    "RandomForest":     y_pred_rf,
})

rmse_by_cluster = (
    per_cluster
    .groupby("SeasonalCluster")
    .apply(lambda g: pd.Series({
        "n":                len(g),
        "Baseline-segment": mean_squared_error(g["y_true"], g["Baseline-segment"]) ** 0.5,
        "Ridge":            mean_squared_error(g["y_true"], g["Ridge"]) ** 0.5,
        "RandomForest":     mean_squared_error(g["y_true"], g["RandomForest"]) ** 0.5,
    }), include_groups=False)
    .round(2)
)

print("RMSE per SeasonalCluster (lower = better):")
print(rmse_by_cluster.to_string())

# RF improvement over the segment baseline, per cluster
rmse_by_cluster["RF_vs_Baseline_%"] = (
    (rmse_by_cluster["RandomForest"] - rmse_by_cluster["Baseline-segment"])
    / rmse_by_cluster["Baseline-segment"] * 100
).round(1)
print("\nNegative % = RandomForest beats the baseline in that tier:")
print(rmse_by_cluster["RF_vs_Baseline_%"].to_string())

RMSE per SeasonalCluster (lower = better):
                     n  Baseline-segment  Ridge  RandomForest
SeasonalCluster                                              
HS               556.0             61.24  50.29         54.01
LS               606.0             23.82  22.95         22.73
S1               127.0             37.34  28.95         33.88
S2               272.0             23.55  22.37         22.22
WTR               54.0             20.97  15.64         14.26

Negative % = RandomForest beats the baseline in that tier:
SeasonalCluster
HS    -11.8
LS     -4.6
S1     -9.3
S2     -5.6
WTR   -32.0


No single model wins everywhere. RF is the best all-round choice (beats baseline in 4/5 tiers), but Ridge is materially better in the two important tiers (HS, S1)